In [1]:
from pyspark.sql import functions as F

# 1. Odczyt surowej tabeli Delta z Lakehouse_Bronze
bronze_gdp_path = "abfss://2f59ccc2-e9d6-44b5-8c0c-1bdbf1fe148a@onelake.dfs.fabric.microsoft.com/33b93b7e-0899-43c6-81a8-654bc740aa32/Tables/dbo/gdp_bronze"

df_raw = spark.read.format("delta").load(bronze_gdp_path)

# 2. Spłaszczenie struktur JSON, rzutowanie typów i odfiltrowanie braków danych
df_silver = (
    df_raw
    .select(
        F.col("countryiso3code").alias("country_iso3"),
        F.col("country.id").alias("country_code"),
        F.col("country.value").alias("country_name"),
        F.col("indicator.id").alias("indicator_id"),
        F.col("indicator.value").alias("indicator_name"),
        F.col("date").cast("int").alias("year"),
        F.col("value").cast("double").alias("gdp_usd")
    )
    .filter(
        F.col("year").isNotNull() &
        F.col("gdp_usd").isNotNull() &
        F.col("country_iso3").isNotNull()
    )
    .dropDuplicates(["country_iso3", "year"])
    .orderBy(F.col("year").desc())
)

# 3. Zapis oczyszczonej tabeli Delta do domyślnego Lakehouse_Silver
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gdp_silver")
)

print(f"Zapisano tabelę gdp_silver. Liczba rekordów: {df_silver.count()}")
display(df_silver.limit(10))

StatementMeta(, dce75306-c885-4de0-8f36-cbb665d201a1, 3, Finished, Available, Finished, False)

Zapisano tabelę gdp_silver. Liczba rekordów: 66


SynapseWidget(Synapse.DataFrame, 812aecbc-ea4f-4c53-a670-bb187aba0886)